# 🎬 Zehon — montage

**Lancé depuis Drive** : `Zehon/montage_zehon.ipynb` n'est qu'un lanceur (le formulaire, puis quelques lignes qui clonent le dépôt et exécutent ce notebook-ci avec `lanceur.py`). Ce notebook ne se recopie jamais dans Drive : une correction poussée sert au lancement suivant.

Il monte le dossier Drive d'une vidéo (`03_scenes.md`, `images/`, `clips/`, `voix/`) avec `monter.py`, et dépose dans `rendu/` :

| Fichier | Contenu |
|---|---|
| `video.mp4` (ou `extrait_AAA-BBB.mp4`) | la vidéo, 1920×1080, son à −14 LUFS |
| `verdict.json` | le verdict court (moins de 1 Ko) : prête ou non, contrôle, avertissements, durée, temps de rendu. **C'est ce que lit Claude** |
| `apercu/` | 6 images réduites : début, titre, plan animé, sous-titres, texte animé, fin |
| `rapport.json` | le détail : chaque scène, les avertissements, le contrôle |
| `sous_titres.srt` | les sous-titres pour YouTube (à ajouter dans YouTube Studio) |

Compte **45 min environ** pour 8 min de vidéo (mesuré : 42 min pour le sel), sans GPU ; `EXTRAIT` = `1-10` pour un essai (environ 5 min).


In [ ]:
# ── Cellule 1 : réglages ────────────────────────────────────────────────────
VIDEO = ""  # @param {type:"string"}
# Vide = la vidéo la plus récente dont la voix est prête. Sinon, le nom du dossier : "01_sel".

SOUS_TITRES = True  # @param {type:"boolean"}
# Coché = les sous-titres incrustés (quelques mots à la fois, le mot dit en couleur).
# Le fichier sous_titres.srt pour YouTube est écrit dans tous les cas.

MUSIQUE = ""  # @param {type:"string"}
# Vide = sans musique. Sinon, le nom d'un fichier rangé dans Zehon/Charte/musique/ : "nappe.mp3".

VOLUME_MUSIQUE_DB = -26  # @param {type:"number"}
# Le volume de la nappe, en dB ; elle baisse encore de 6 dB quand la voix parle.

EXTRAIT = ""  # @param {type:"string"}
# Vide = la vidéo entière. "1-10" = un essai sur les scènes 1 à 10.

RACINE = "/content/drive/MyDrive/Zehon"  # @param {type:"string"}
# Le dossier Zehon dans ton Drive. Si Zehon est « partagé avec moi », ajoute-lui un raccourci dans « Mon Drive ».

BRANCHE = "claude/zelan-studio-analysis-rws9f6"  # @param {type:"string"}
# La branche du dépôt GitHub d'où vient le code du montage.

In [ ]:
# ── Cellule 2 : Drive, outils et code du montage ─────────────────────────────
import json, os, re, shutil, subprocess, sys, time
from pathlib import Path

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

RACINE = Path(RACINE)
if not RACINE.is_dir():
    candidats = [p for p in Path("/content/drive/MyDrive").glob("**/Zehon") if p.is_dir()][:3]
    raise SystemExit(f"❌ Dossier Zehon introuvable : {RACINE}\n   Candidats trouvés : {candidats or 'aucun'}\n"
                     "   → Corrige RACINE dans la cellule 1.")
print(f"✓ Zehon : {RACINE}")

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
for module, paquet in [("cv2", "opencv-python-headless"), ("PIL", "pillow"), ("numpy", "numpy")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], check=True)
print("✓ ffmpeg, OpenCV, Pillow, numpy")

# Une seule source pour le code : le dépôt, cloné à neuf à chaque lancement (par le lanceur Drive).
DEPOT = Path("/content/Channel")
if not globals().get("DEPOT_PRET"):
    shutil.rmtree(DEPOT, ignore_errors=True)
    clone = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE,
                            "https://github.com/Johnfrancis6/Channel.git", str(DEPOT)], capture_output=True, text=True)
    if clone.returncode != 0:
        raise SystemExit(f"❌ Impossible de récupérer le code (branche « {BRANCHE} ») :\n{clone.stderr[-500:]}")
COMMIT = subprocess.run(["git", "-C", str(DEPOT), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
MONTER = DEPOT / "zehon" / "video-maker" / "scripts" / "monter.py"
print(f"✓ Code du montage : {BRANCHE} @ {COMMIT}")

In [ ]:
# ── Cellule 3 : la vidéo et le plan ─────────────────────────────────────────
VIDEOS = RACINE / "Videos"

def voix_prete(d):
    try:
        return json.loads((d / "voix" / "etat.json").read_text(encoding="utf-8")).get("statut") == "pret"
    except Exception:
        return False

if VIDEO:
    DOSSIER = VIDEOS / VIDEO
    if not DOSSIER.is_dir():
        raise SystemExit(f"❌ {DOSSIER} introuvable. Vidéos : {[d.name for d in sorted(VIDEOS.iterdir()) if d.is_dir()]}")
else:
    pretes = [d for d in sorted(VIDEOS.iterdir()) if d.is_dir() and voix_prete(d)]
    if not pretes:
        raise SystemExit("❌ Aucune vidéo n'a sa voix prête : lance d'abord voix_zehon.ipynb.")
    DOSSIER = pretes[-1]
if not voix_prete(DOSSIER):
    raise SystemExit(f"❌ La voix de {DOSSIER.name} n'est pas prête (voix/etat.json) : lance d'abord voix_zehon.ipynb.")
print(f"🎬 Vidéo : {DOSSIER.name}\n")

def lancer(*options):
    """Lance monter.py sur la vidéo et affiche ce qu'il dit au fil de l'eau. Renvoie son code de sortie."""
    p = subprocess.Popen([sys.executable, "-u", str(MONTER), str(DOSSIER), *options],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ligne in p.stdout:
        print(ligne, end="")
    return p.wait()

if lancer("--plan") != 0:
    raise SystemExit("\n❌ Le plan a échoué : corrige ce qui est dit juste au-dessus, puis relance.")

In [ ]:
# ── Cellule 4 : le rendu ────────────────────────────────────────────────────
# La vidéo se fabrique sur le disque de Colab (plus rapide et plus sûr), puis est copiée dans Drive.
LOCAL = Path("/content/rendu")
shutil.rmtree(LOCAL, ignore_errors=True)
LOCAL.mkdir()
if EXTRAIT:
    a, _, b = EXTRAIT.strip().partition("-")
    nom = f"extrait_{int(a):03d}-{int(b or a):03d}"
else:
    nom = "video"
SORTIE = LOCAL / f"{nom}.mp4"
RAPPORT = LOCAL / ("rapport.json" if nom == "video" else f"rapport_{nom}.json")

options = ["--sortie", str(SORTIE)]
if SOUS_TITRES:
    options.append("--sous-titres")
if EXTRAIT:
    options += ["--scenes", EXTRAIT.strip()]
if MUSIQUE:
    nappe = Path(MUSIQUE) if os.path.isabs(MUSIQUE) else RACINE / "Charte" / "musique" / MUSIQUE
    if not nappe.is_file():
        raise SystemExit(f"❌ Musique introuvable : {nappe}")
    options += ["--musique", str(nappe), "--volume-musique", str(VOLUME_MUSIQUE_DB)]

code = lancer(*options)
if not RAPPORT.is_file():
    raise SystemExit("\n❌ Le rendu s'est arrêté avant la fin : relis les messages ci-dessus.")

# monter.py a écrit le rapport, verdict.json et apercu/ à côté de la vidéo : tout part dans Drive.
DESTINATION = DOSSIER / "rendu"
DESTINATION.mkdir(exist_ok=True)
rapport = json.loads(RAPPORT.read_text(encoding="utf-8"))
rapport["sortie"] = str(DESTINATION / SORTIE.name)
RAPPORT.write_text(json.dumps(rapport, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n⏱️  Rendu en {rapport['rendu_s'] / 60:.1f} min. Copie dans Drive…")
for fichier in (SORTIE, RAPPORT, LOCAL / "verdict.json"):
    shutil.copy2(fichier, DESTINATION / fichier.name)
    print(f"✓ {DESTINATION / fichier.name}")
shutil.rmtree(DESTINATION / "apercu", ignore_errors=True)
shutil.copytree(LOCAL / "apercu", DESTINATION / "apercu")
print(f"✓ {DESTINATION / 'apercu'} ({len(list((LOCAL / 'apercu').glob('*.jpg')))} images)")
if not EXTRAIT:
    print(f"✓ {DESTINATION / 'sous_titres.srt'}")


In [ ]:
# ── Cellule 5 : le verdict ──────────────────────────────────────────────────
verdict = json.loads((LOCAL / "verdict.json").read_text(encoding="utf-8"))
for ligne in verdict["controle"] + [f"⚠️  {a}" for a in verdict["avertissements"]]:
    print(ligne)
if verdict["pret"]:
    print(f"\n✅ {SORTIE.name} est dans Drive et passe le contrôle. Dis à Claude : « lis le verdict ».")
else:
    print("\n❌ Le contrôle a échoué : ne pas publier en l'état. Dis à Claude : « lis le verdict ».")

try:  # s'assurer que Drive a bien reçu la vidéo avant de fermer l'onglet
    from google.colab import drive
    drive.flush_and_unmount()
    print("✓ Drive synchronisé : tu peux fermer l'onglet.")
except ImportError:
    pass
